# 국민문화예술활동조사 파생변수 추가 실험

- 실험명: culture_art_feature_experiment
- 목적: 국민문화예술활동조사의 집단별 문화예술 관여도 변수를 선호도 예측 모델에 추가함
- 모델: Multinomial Logistic, HistGradientBoosting
- 산출물: 노트북 출력으로 성능 비교만 확인함

## 1. 패키지 및 경로

- 국민여가활동조사 기반 선호도 학습 데이터를 불러옴.
- 국민문화예술활동조사 선택 칼럼 정리본을 함께 불러옴.
- 별도 CSV 파일은 생성하지 않음.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    top_k_accuracy_score,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

BASE_PATH = Path.cwd()
while BASE_PATH.name != "oracle_mnc_project" and BASE_PATH.parent != BASE_PATH:
    BASE_PATH = BASE_PATH.parent

LEISURE_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "source" / "leisure_activity_survey_2021_2025_selected_columns_enriched.csv"
MAPPING_PATH = BASE_PATH / "notebooks" / "preference" / "data" / "processed" / "satisfaction" / "ml_activity_category_mapping.csv"
CULTURE_ART_PATH = BASE_PATH / "data" / "raw" / "preferences" / "source" / "culture_art_activity_survey" / "culture_art_activity_survey_2021_2025_selected_columns.csv"

print("LEISURE_PATH 존재:", LEISURE_PATH.exists())
print("MAPPING_PATH 존재:", MAPPING_PATH.exists())
print("CULTURE_ART_PATH 존재:", CULTURE_ART_PATH.exists())

## 2. 국민문화예술활동조사 파생변수 설계

- 직접관람 횟수: 실제 문화예술 경험 수준으로 사용함.
- 향후 직접관람 의향: 향후 문화예술 소비 의향으로 사용함.
- 문화공간 이용 여부: 문화시설 이용 경험으로 사용함.
- 문화예술 지출: 문화예술 소비 강도로 사용함.
- 공연계열 의향은 연극·뮤지컬·무용·전통예술 중심으로 산정함.

In [ ]:
culture_art = pd.read_csv(CULTURE_ART_PATH, encoding="utf-8-sig")

direct_cols = [c for c in culture_art.columns if c.startswith("문화예술행사 직접관람 횟수_")]
intent_cols = [c for c in culture_art.columns if c.startswith("향후 1년 이내 직접관람 의향_")]
space_cols = [c for c in culture_art.columns if c.startswith("문화예술활동 공간 이용여부_")]
future_spend_cols = [c for c in culture_art.columns if c.startswith("향후 지출을 늘리고 싶은 항목_")]

expense_cols = [
    "문화예술활동 직접관람 비용_총 합계",
    "문화예술활동 구매 및 대여 비용_총 합계",
    "문화예술활동 참여 비용_총 합계",
    "문화예술활동 교육 비용_총 합계",
    "문화예술활동 월평균 합계_총 합계",
]

for col in direct_cols + intent_cols + space_cols + future_spend_cols + expense_cols:
    culture_art[col] = pd.to_numeric(culture_art[col], errors="coerce")

culture_art["art_직접관람횟수합"] = culture_art[direct_cols].fillna(0).sum(axis=1)
culture_art["art_직접관람경험"] = (culture_art["art_직접관람횟수합"] > 0).astype(int)
culture_art["art_향후관람의향수"] = (culture_art[intent_cols] == 1).sum(axis=1)
culture_art["art_향후관람의향있음"] = (culture_art["art_향후관람의향수"] > 0).astype(int)
culture_art["art_문화공간이용수"] = (culture_art[space_cols] == 1).sum(axis=1)
culture_art["art_문화공간이용있음"] = (culture_art["art_문화공간이용수"] > 0).astype(int)
culture_art["art_향후지출증가의향수"] = (culture_art[future_spend_cols] == 1).sum(axis=1)

genre_intent_map = {
    "art_문학관람의향": ["향후 1년 이내 직접관람 의향_문학행사"],
    "art_미술관람의향": ["향후 1년 이내 직접관람 의향_미술전시회"],
    "art_영상관람의향": ["향후 1년 이내 직접관람 의향_영화"],
    "art_공연예술관람의향": [
        "향후 1년 이내 직접관람 의향_전통예술",
        "향후 1년 이내 직접관람 의향_연극",
        "향후 1년 이내 직접관람 의향_뮤지컬",
        "향후 1년 이내 직접관람 의향_무용",
    ],
}

for new_col, cols in genre_intent_map.items():
    valid_cols = [c for c in cols if c in culture_art.columns]
    culture_art[new_col] = ((culture_art[valid_cols] == 1).sum(axis=1) > 0).astype(int)

general_art_features = [
    "art_직접관람경험",
    "art_직접관람횟수합",
    "art_향후관람의향있음",
    "art_향후관람의향수",
    "art_문화공간이용있음",
    "art_문화공간이용수",
    "art_향후지출증가의향수",
    "문화예술활동 월평균 합계_총 합계",
]

genre_art_features = list(genre_intent_map.keys())
art_feature_cols = general_art_features + genre_art_features

print("국민문화예술활동조사 구조:", culture_art.shape)
print("사용 파생변수 수:", len(art_feature_cols))
display(culture_art[["조사년도", "성별", "연령", "가구소득", "지역규모", "17개 시도"] + art_feature_cols].head())

## 3. 집단별 문화예술 파생변수 생성

- 두 조사는 같은 응답자가 아니므로 개인 ID로 직접 결합하지 않음.
- 조사년도·성별·연령·소득·시도·지역규모가 같은 집단의 가중평균을 생성함.
- 정확 매칭이 없으면 더 넓은 집단 평균으로 순차 보완함.

In [ ]:
def to_code_string(series):
    return pd.to_numeric(series, errors="coerce").astype("Int64").astype("string")

culture_art["가구소득_생성"] = culture_art["가구소득"]
key_cols_exact = ["조사년도", "성별", "연령", "가구소득_생성", "17개 시도", "지역규모"]

for col in key_cols_exact:
    culture_art[col] = to_code_string(culture_art[col])

culture_art["최종가중치"] = pd.to_numeric(culture_art["최종가중치"], errors="coerce").fillna(1)
culture_art["최종가중치"] = culture_art["최종가중치"].where(culture_art["최종가중치"] > 0, 1)

def make_weighted_lookup(df, keys, feature_cols, weight_col="최종가중치"):
    temp = df[keys + feature_cols + [weight_col]].dropna(subset=keys).copy()
    temp[feature_cols] = temp[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0)

    weighted = temp[feature_cols].multiply(temp[weight_col], axis=0)
    weighted[keys] = temp[keys]
    weighted[weight_col] = temp[weight_col]

    sum_weighted = weighted.groupby(keys, as_index=False)[feature_cols].sum()
    sum_weight = temp.groupby(keys, as_index=False).agg(
        art_집계표본수=(weight_col, "size"),
        art_가중치합=(weight_col, "sum"),
    )

    lookup = sum_weighted.merge(sum_weight, on=keys, how="left")
    for col in feature_cols:
        lookup[col] = lookup[col] / lookup["art_가중치합"]

    return lookup.drop(columns="art_가중치합")

fallback_key_sets = [
    ("exact", ["조사년도", "성별", "연령", "가구소득_생성", "17개 시도", "지역규모"]),
    ("drop_region_size", ["조사년도", "성별", "연령", "가구소득_생성", "17개 시도"]),
    ("drop_city_region", ["조사년도", "성별", "연령", "가구소득_생성"]),
    ("drop_income", ["조사년도", "성별", "연령"]),
    ("year_only", ["조사년도"]),
]

lookups = {
    level: make_weighted_lookup(culture_art, keys, art_feature_cols)
    for level, keys in fallback_key_sets
}

print("lookup 집단 수")
for level, lookup in lookups.items():
    print(level, lookup.shape)

## 4. 선호도 학습 테이블 생성

- 국민여가활동조사의 향후 희망 여가활동 1~3순위를 사용함.
- 순위 가중치는 1순위 1.5, 2순위 1.25, 3순위 1.0으로 적용함.
- 교통수단·여행사·음악·체육용품은 모델 타깃에서 제외함.

In [ ]:
raw = pd.read_csv(LEISURE_PATH, encoding="utf-8-sig")
mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")
mapping.columns = ["activity_code", "activity_name", "category", "use_target"]

# 기준 중위소득 100%, 원/월
median_income_100 = {
    2024: {
        1: 2228445, 2: 3682609, 3: 4714657, 4: 5729913,
        5: 6695735, 6: 7618369, 7: 8514994,
    },
    2025: {
        1: 2392013, 2: 3932658, 3: 5025353, 4: 6097773,
        5: 7108192, 6: 8064805, 7: 8988428,
    },
}

income_mid_10k = {
    1: 50,
    2: 150,
    3: 250,
    4: 350,
    5: 450,
    6: 550,
    7: 650,
}

def household_size_cap(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x < 1:
        return np.nan
    return min(x, 7)

def median_ratio(row):
    year = int(row["조사년도"])
    size = household_size_cap(row["동거가구원수_생성"])
    income_code = row["가구소득_생성"]

    if pd.isna(size) or pd.isna(income_code):
        return np.nan

    income_won = income_mid_10k.get(int(income_code), np.nan) * 10000
    median_won = median_income_100.get(year, {}).get(size, np.nan)

    if pd.isna(income_won) or pd.isna(median_won) or median_won == 0:
        return np.nan

    return income_won / median_won

raw["중위소득비율_근사"] = raw.apply(median_ratio, axis=1)
raw["소득구간_다층"] = pd.cut(
    raw["중위소득비율_근사"],
    bins=[-np.inf, 0.5, 1.0, 1.5, np.inf],
    labels=["50이하", "50_100", "100_150", "150초과"],
).astype(str)
raw.loc[raw["중위소득비율_근사"].isna(), "소득구간_다층"] = "unknown"

excluded_categories = ["분류범위외", "교통수단", "여행사", "음악", "체육용품"]
mapping["use_target_final"] = (
    mapping["use_target"].astype(bool)
    & ~mapping["category"].isin(excluded_categories)
)

rank_cols = {
    1: "향후 희망하는 여가활동 1순위",
    2: "향후 희망하는 여가활동 2순위",
    3: "향후 희망하는 여가활동 3순위",
}
rank_score = {1: 1.5, 2: 1.25, 3: 1.0}

base_cat_cols = [
    "성별",
    "연령",
    "조사년도",
    "성별_연령",
    "소득구간_다층",
    "장애여부",
    "17개 시도",
]

preference_raw = raw.loc[
    raw["조사년도"].isin([2024, 2025]),
    [
        "응답자_ID", "최종가중치", "성별", "연령", "조사년도",
        "소득구간_다층", "가구소득_생성", "장애여부", "17개 시도", "지역규모",
    ] + list(rank_cols.values())
].copy()

preference_raw["성별_연령"] = (
    preference_raw["성별"].astype("Int64").astype(str)
    + "_"
    + preference_raw["연령"].astype("Int64").astype(str)
)

for col in ["조사년도", "성별", "연령", "가구소득_생성", "17개 시도", "지역규모"]:
    preference_raw[col] = to_code_string(preference_raw[col])

for col in base_cat_cols:
    preference_raw[col] = preference_raw[col].astype("string").fillna("unknown")

print("선호도 원자료 구조:", preference_raw.shape)
print("2024~2025 응답자 수:", preference_raw["응답자_ID"].nunique())

## 5. 국민문화예술활동조사 파생변수 결합

- 정확 매칭부터 시도함.
- 결측이 남는 응답자는 지역규모 제외, 시도 제외, 소득 제외, 연도 평균 순서로 보완함.
- 최종적으로 모든 응답자에 문화예술 파생변수를 부여함.

In [ ]:
def attach_with_fallback(base_df, lookups, fallback_key_sets, feature_cols):
    result = base_df.copy()
    fill_cols = feature_cols + ["art_집계표본수"]

    for col in fill_cols:
        result[col] = np.nan

    result["art_매칭수준"] = pd.NA

    match_summary = []

    for level, keys in fallback_key_sets:
        lookup = lookups[level]
        merge_cols = keys + fill_cols

        missing_mask = result[feature_cols[0]].isna()
        before_missing = missing_mask.sum()

        if before_missing == 0:
            match_summary.append({
                "매칭수준": level,
                "보완전결측": 0,
                "보완건수": 0,
                "보완후결측": 0,
            })
            continue

        temp = result.loc[missing_mask, ["응답자_ID"] + keys].merge(
            lookup[merge_cols],
            on=keys,
            how="left",
        )
        temp = temp.set_index("응답자_ID")
        fill_index = temp.index[temp[feature_cols[0]].notna()]

        for col in fill_cols:
            result.loc[result["응답자_ID"].isin(fill_index), col] = (
                result.loc[result["응답자_ID"].isin(fill_index), "응답자_ID"]
                .map(temp[col])
                .to_numpy()
            )

        result.loc[result["응답자_ID"].isin(fill_index), "art_매칭수준"] = level
        after_missing = result[feature_cols[0]].isna().sum()

        match_summary.append({
            "매칭수준": level,
            "보완전결측": int(before_missing),
            "보완건수": int(before_missing - after_missing),
            "보완후결측": int(after_missing),
        })

    for col in fill_cols:
        if result[col].isna().any():
            result[col] = result[col].fillna(result[col].median())

    result["art_매칭수준"] = result["art_매칭수준"].fillna("median_fill")

    return result, pd.DataFrame(match_summary)

preference_art, art_match_summary = attach_with_fallback(
    preference_raw,
    lookups,
    fallback_key_sets,
    art_feature_cols,
)

print("문화예술 파생변수 매칭 결과")
display(art_match_summary)

print("최종 결측")
print(preference_art[art_feature_cols + ["art_집계표본수"]].isna().sum().to_string())

print("매칭수준 분포")
print(preference_art["art_매칭수준"].value_counts().to_string())

## 6. 순위형 학습용 long 테이블 생성

- 유효한 문화누리 중분류만 남김.
- 동일 응답자의 1~3순위 안에서 중복 중분류는 한 번만 사용함.
- 응답자 내부 순위 가중치를 정규화한 뒤 최종가중치를 곱함.

In [ ]:
code_to_category = mapping.set_index("activity_code")["category"].to_dict()
code_to_use = mapping.set_index("activity_code")["use_target_final"].to_dict()

wide_records = []
long_records = []

for _, row in preference_art.iterrows():
    valid_rows = []
    seen_categories = set()

    for rank_no, col in rank_cols.items():
        activity_code = row[col]
        if pd.isna(activity_code):
            continue

        activity_code = int(activity_code)
        category = code_to_category.get(activity_code)
        use_target = bool(code_to_use.get(activity_code, False))

        if not use_target:
            continue

        if category in seen_categories:
            continue

        seen_categories.add(category)
        valid_rows.append({
            "rank_no": rank_no,
            "rank_score": rank_score[rank_no],
            "target_category": category,
        })

    score_sum = sum(x["rank_score"] for x in valid_rows)

    wide_row = {
        "응답자_ID": row["응답자_ID"],
        "성별": row["성별"],
        "연령": row["연령"],
        "조사년도": row["조사년도"],
        "성별_연령": row["성별_연령"],
        "소득구간_다층": row["소득구간_다층"],
        "장애여부": row["장애여부"],
        "17개 시도": row["17개 시도"],
        "최종가중치": row["최종가중치"],
        "선호_유효순위수": len(valid_rows),
    }
    for col in art_feature_cols + ["art_집계표본수"]:
        wide_row[col] = row[col]

    for i, valid in enumerate(valid_rows, start=1):
        wide_row[f"선호_유효중분류_{i}순위"] = valid["target_category"]
        long_record = {
            "응답자_ID": row["응답자_ID"],
            "성별": row["성별"],
            "연령": row["연령"],
            "조사년도": row["조사년도"],
            "성별_연령": row["성별_연령"],
            "소득구간_다층": row["소득구간_다층"],
            "장애여부": row["장애여부"],
            "17개 시도": row["17개 시도"],
            "rank_no": valid["rank_no"],
            "rank_score": valid["rank_score"],
            "target_category": valid["target_category"],
            "sample_weight": row["최종가중치"] * valid["rank_score"] / score_sum,
        }
        for col in art_feature_cols + ["art_집계표본수"]:
            long_record[col] = row[col]
        long_records.append(long_record)

    wide_records.append(wide_row)

rank_base = pd.DataFrame(wide_records)
rank_long = pd.DataFrame(long_records)

print("응답자 테이블:", rank_base.shape)
print("순위 long 테이블:", rank_long.shape)
print("유효 순위 수")
print(rank_base["선호_유효순위수"].value_counts().sort_index().to_string())

## 7. Split 및 모델 함수

- train/test = 8:2
- train 내부 train/valid = 8:2
- 조사년도와 1순위 타깃을 함께 고려해 stratify함.
- 3-Fold Stratified CV를 함께 수행함.

In [ ]:
primary_target = (
    rank_long.sort_values(["응답자_ID", "rank_no"])
    .groupby("응답자_ID", as_index=False)["target_category"]
    .first()
    .rename(columns={"target_category": "primary_target"})
)

model_base = (
    rank_base.loc[rank_base["선호_유효순위수"] > 0]
    .merge(primary_target, on="응답자_ID", how="left")
    .copy()
)

def choose_strata(df, min_count=2):
    year_target = df["조사년도"].astype(str) + "_" + df["primary_target"].astype(str)
    if year_target.value_counts().min() >= min_count:
        return year_target

    target_only = df["primary_target"].astype(str)
    if target_only.value_counts().min() >= min_count:
        return target_only

    return None

train_valid_base, test_base = train_test_split(
    model_base,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(model_base, 2),
)

train_base, valid_base = train_test_split(
    train_valid_base,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=choose_strata(train_valid_base, 2),
)

train_data = rank_long[rank_long["응답자_ID"].isin(train_base["응답자_ID"])].copy()
valid_data = rank_long[rank_long["응답자_ID"].isin(valid_base["응답자_ID"])].copy()
test_data = rank_long[rank_long["응답자_ID"].isin(test_base["응답자_ID"])].copy()

classes = np.array(sorted(rank_long["target_category"].unique()))

split_summary = pd.DataFrame({
    "dataset": ["train", "valid", "test"],
    "respondents": [
        train_base["응답자_ID"].nunique(),
        valid_base["응답자_ID"].nunique(),
        test_base["응답자_ID"].nunique(),
    ],
})
split_summary["share"] = split_summary["respondents"] / model_base["응답자_ID"].nunique()
display(split_summary)

def make_models(cat_cols, num_cols):
    preprocess = ColumnTransformer(
        [
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
            ("num", StandardScaler(), num_cols),
        ],
        remainder="drop",
    )

    return {
        "multinomial_logistic": Pipeline([
            ("preprocess", preprocess),
            ("model", LogisticRegression(max_iter=1000, solver="lbfgs", C=1.0))
        ]),
        "hist_gradient_boosting": Pipeline([
            ("preprocess", preprocess),
            ("model", HistGradientBoostingClassifier(
                max_iter=60,
                learning_rate=0.06,
                max_leaf_nodes=8,
                l2_regularization=1.0,
                random_state=42,
            ))
        ]),
    }

def fit_model(model, data, cat_cols, num_cols):
    x = data[cat_cols + num_cols].copy()
    x[cat_cols] = x[cat_cols].astype(str)
    x[num_cols] = x[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    y = data["target_category"]
    w = data["sample_weight"]
    return model.fit(x, y, model__sample_weight=w)

def align_proba(model, proba):
    model_classes = model.named_steps["model"].classes_
    aligned = np.zeros((proba.shape[0], len(classes)))
    class_to_idx = {c: i for i, c in enumerate(model_classes)}

    for j, c in enumerate(classes):
        if c in class_to_idx:
            aligned[:, j] = proba[:, class_to_idx[c]]

    row_sum = aligned.sum(axis=1, keepdims=True)
    return np.divide(aligned, row_sum, out=np.zeros_like(aligned), where=row_sum > 0)

def weighted_brier(y_true, proba, sample_weight):
    y_index = pd.Categorical(y_true, categories=classes).codes
    y_onehot = np.zeros_like(proba)
    y_onehot[np.arange(len(y_index)), y_index] = 1
    return np.average(((proba - y_onehot) ** 2).sum(axis=1), weights=sample_weight)

def evaluate(experiment, model_name, model, data, dataset_name, cat_cols, num_cols):
    x = data[cat_cols + num_cols].copy()
    x[cat_cols] = x[cat_cols].astype(str)
    x[num_cols] = x[num_cols].apply(pd.to_numeric, errors="coerce").fillna(0)
    y = data["target_category"].to_numpy()
    w = data["sample_weight"].to_numpy()
    proba = align_proba(model, model.predict_proba(x))
    y_pred = classes[np.argmax(proba, axis=1)]

    return {
        "experiment": experiment,
        "model": model_name,
        "dataset": dataset_name,
        "LogLoss": log_loss(y, proba, labels=classes, sample_weight=w),
        "Top1_Accuracy": accuracy_score(y, y_pred, sample_weight=w),
        "Top3_HitRate": top_k_accuracy_score(y, proba, k=3, labels=classes, sample_weight=w),
        "Macro_F1": f1_score(y, y_pred, labels=classes, average="macro", sample_weight=w, zero_division=0),
        "Weighted_F1": f1_score(y, y_pred, labels=classes, average="weighted", sample_weight=w, zero_division=0),
        "Balanced_Accuracy": balanced_accuracy_score(y, y_pred),
        "Brier": weighted_brier(y, proba, w),
    }

## 8. 모델 실험

- baseline: 소득 다층 + 시도 + 장애여부
- culture_general: baseline + 문화예술 일반 관여도 파생변수
- culture_general_genre: culture_general + 장르별 관람 의향 파생변수

In [ ]:
experiments = {
    "income_band_sido_disability": {
        "cat_cols": base_cat_cols,
        "num_cols": [],
    },
    "culture_general": {
        "cat_cols": base_cat_cols,
        "num_cols": general_art_features + ["art_집계표본수"],
    },
    "culture_general_genre": {
        "cat_cols": base_cat_cols,
        "num_cols": general_art_features + genre_art_features + ["art_집계표본수"],
    },
}

performance_rows = []
cv_rows = []

cv_base = train_valid_base.reset_index(drop=True)
cv_strata = choose_strata(cv_base, min_count=3)
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

for exp_name, cols in experiments.items():
    cat_cols = cols["cat_cols"]
    num_cols = cols["num_cols"]

    for fold_no, (tr_idx, va_idx) in enumerate(skf.split(cv_base, cv_strata), start=1):
        tr_ids = cv_base.iloc[tr_idx]["응답자_ID"]
        va_ids = cv_base.iloc[va_idx]["응답자_ID"]

        fold_train_data = rank_long[rank_long["응답자_ID"].isin(tr_ids)].copy()
        fold_valid_data = rank_long[rank_long["응답자_ID"].isin(va_ids)].copy()

        for model_name, model in make_models(cat_cols, num_cols).items():
            print("cv", exp_name, fold_no, model_name)
            model = fit_model(model, fold_train_data, cat_cols, num_cols)
            row = evaluate(exp_name, model_name, model, fold_valid_data, "cv_valid", cat_cols, num_cols)
            row["fold"] = fold_no
            cv_rows.append(row)

    for model_name, model in make_models(cat_cols, num_cols).items():
        print("holdout", exp_name, model_name)
        model = fit_model(model, train_data, cat_cols, num_cols)

        for dataset_name, dataset in [
            ("train", train_data),
            ("valid", valid_data),
            ("test", test_data),
        ]:
            performance_rows.append(
                evaluate(exp_name, model_name, model, dataset, dataset_name, cat_cols, num_cols)
            )

cv_performance = pd.DataFrame(cv_rows)
performance = pd.DataFrame(performance_rows)

cv_summary = cv_performance.groupby(["experiment", "model"], as_index=False).agg(
    LogLoss_mean=("LogLoss", "mean"),
    LogLoss_std=("LogLoss", "std"),
    Top1_Accuracy_mean=("Top1_Accuracy", "mean"),
    Top3_HitRate_mean=("Top3_HitRate", "mean"),
    Macro_F1_mean=("Macro_F1", "mean"),
    Balanced_Accuracy_mean=("Balanced_Accuracy", "mean"),
    Brier_mean=("Brier", "mean"),
)

test_summary = performance[performance["dataset"] == "test"].sort_values("LogLoss").copy()

print("3-Fold CV")
display(cv_summary.sort_values("LogLoss_mean"))

print("Holdout Test")
display(test_summary)

## 9. 결과 해석용 요약

- LogLoss와 Brier는 확률 예측 품질을 봄.
- Top1·Top3는 가장 높은 확률의 분류가 실제 응답에 들어가는지 봄.
- Macro F1과 Balanced Accuracy는 소수 분류까지 균형 있게 맞추는지 봄.

In [ ]:
baseline_test = test_summary[
    test_summary["experiment"].eq("income_band_sido_disability")
].copy()

best_test = test_summary.iloc[0].copy()

print("기준 모델 test 성능")
display(baseline_test)

print("최고 LogLoss 모델")
display(pd.DataFrame([best_test]))

metric_cols = ["LogLoss", "Top1_Accuracy", "Top3_HitRate", "Macro_F1", "Balanced_Accuracy", "Brier"]
compare = test_summary[["experiment", "model"] + metric_cols].copy()
display(compare)

## 10. ?? ?? ??

- ?????????? ????? ??? ID? ?? ???? ??, ??????????????????????? ?? ??? ?????? ???.
- ?? ?? 18,475?, ???? ?? ?? 670?, ??????? ?? ?? 958??? ?? ?? ?? ???.
- LogLoss ?? ?? ??? ?? `income_band_sido_disability + multinomial_logistic`?.
- ???? ???? ??? Top1, Macro F1? ?? ????? LogLoss? Brier? ???.
- H3SFCA ????? ??? ??? ?? ??? ?????, ?? ??? ?? ???? ????? ? ???? ?? ?? ??? ???.

### Holdout Test ??

| experiment | model | LogLoss | Top1 | Top3 | Macro F1 | Balanced Acc | Brier |
|---|---|---:|---:|---:|---:|---:|---:|
| income_band_sido_disability | multinomial_logistic | 1.701063 | 0.324676 | 0.735235 | 0.115518 | 0.142826 | 0.778067 |
| income_band_sido_disability | hist_gradient_boosting | 1.701306 | 0.323194 | 0.738515 | 0.110422 | 0.140081 | 0.777974 |
| culture_general | multinomial_logistic | 1.704320 | 0.327340 | 0.734669 | 0.118834 | 0.143355 | 0.778529 |
| culture_general_genre | multinomial_logistic | 1.704844 | 0.326685 | 0.735155 | 0.120465 | 0.143974 | 0.778820 |
| culture_general | hist_gradient_boosting | 1.712986 | 0.321098 | 0.728729 | 0.110059 | 0.139229 | 0.780878 |
| culture_general_genre | hist_gradient_boosting | 1.713164 | 0.322927 | 0.732879 | 0.110552 | 0.139053 | 0.780984 |